# Exact robustness certificates for Hinglish classifiers under romanized spelling variation: full experiments

This notebook produces every table and figure in the paper.

- **Resumable.** Each run (dataset × model × canonicalizer × seed) is saved as soon as it finishes, and finished runs are skipped. Set `USE_DRIVE = True` to keep results across Colab sessions.
- **Runtime.** About 2.5–3 hours on a T4 for the full grid. You can run a subset by editing `RUNS`.
- **Final step.** The last section aggregates everything into `paper_results.zip`, containing CSV and LaTeX tables, figures and the variant lexicon. Upload that file to the chat.

**Threat models** (only Hindi-tagged tokens are rewritten, using gold word-level tags):
- **R:** spellings attested in Dakshina for the same Devanagari word.
- **R+:** R plus five deterministic phonetic rules (final *n*, long vowels, aspirate *h*, *w/v* and *z/j*, final *h*). Generated spellings are attached to the most-attested class and never merge two attested classes.

**Canonicalizers:**
- `raw`: no canonicalizer.
- `lexR`: exact for R.
- `lexRp`: exact for R and R+.
- `lex_train`: built from the train lexicon only, so it leaks. It demonstrates the hybrid certificate and the label-free audit.

In [ ]:
!pip -q install -U transformers accelerate scipy pandas matplotlib scikit-learn

## 0. Settings

In [ ]:
USE_DRIVE = False                     # True: mount Google Drive and keep results across sessions
MODELS = {"hingroberta": "l3cube-pune/hing-roberta", "xlmr": "xlm-roberta-base", "muril": "google/muril-base-cased"}
DATASETS = ["sentimix", "sarcasm"]
CANONS = ["raw", "lexR", "lexRp", "lex_train"]
SEEDS_MAIN, SEEDS_OTHER = [0, 1, 2], [0]       # 3 seeds for HingRoBERTa, 1 for the others
RUNS = [(d, m, c, s) for d in DATASETS for m in MODELS for c in CANONS for s in (SEEDS_MAIN if m == "hingroberta" else SEEDS_OTHER)]
MAX_LEN, BS, LR = 64, 32, 2e-5
EPOCHS = {"sentimix": 3, "sarcasm": 4}
ALPHA, CAP = 0.10, 256                # conformal level; max orbit images enumerated exactly
SENTIMIX_URL = "https://zenodo.org/records/3974927/files/Semeval_2020_task9_data.zip?download=1"
DAKSHINA_URL = "https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar"
SARC_BASE = "https://raw.githubusercontent.com/sahilswami96/SarcasmDetection_CodeMixed/master/Dataset/"

import os, io, re, json, math, random, time, zipfile, tarfile, urllib.request, itertools, collections
import numpy as np, pandas as pd, torch
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); ROOT = "/content/drive/MyDrive/hinglish_certified"
else:
    ROOT = "hinglish_certified"
DATA_DIR, OUT_DIR = f"{ROOT}/data", f"{ROOT}/results"
for d in (DATA_DIR, f"{OUT_DIR}/runs", f"{OUT_DIR}/tables", f"{OUT_DIR}/figures", f"{OUT_DIR}/lexicon"): os.makedirs(d, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, "|", len(RUNS), "runs planned")

## 1. Data

In [ ]:
MIRRORS = {SENTIMIX_URL: [SENTIMIX_URL, "https://zenodo.org/api/records/3974927/files/Semeval_2020_task9_data.zip/content",
                          "https://zenodo.org/record/3974927/files/Semeval_2020_task9_data.zip?download=1"]}
def fetch(url, path, tries=6):
    # retries with back-off and mirrors; downloads to a temp file and validates zips before keeping them
    if os.path.exists(path) and (not path.endswith(".zip") or zipfile.is_zipfile(path)): return path
    last = None
    for attempt in range(tries):
        for u in MIRRORS.get(url, [url]):
            try:
                print(f"downloading (attempt {attempt+1}) {u}")
                req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=120) as r, open(path + ".part", "wb") as f:
                    while True:
                        chunk = r.read(1 << 20)
                        if not chunk: break
                        f.write(chunk)
                if path.endswith(".zip") and not zipfile.is_zipfile(path + ".part"): raise IOError("not a valid zip")
                os.replace(path + ".part", path); return path
            except Exception as e:
                last = e; print("  failed:", e)
        time.sleep(min(60, 5 * 2 ** attempt))
    raise RuntimeError(f"Could not download {url}: {last}. Download it manually and upload it to {path}")

def is_hin(tag): return tag.lower() in ("hin", "hi", "hindi")
def is_eng(tag): return tag.lower() in ("eng", "en", "english")

# ---- SentiMix (CC BY 4.0) ----
SENT_MAP = {"positive": 2, "neutral": 1, "negative": 0}
def parse_conll(text):
    out, cur = [], None
    for line in text.splitlines():
        parts = line.strip().split("\t") if "\t" in line else line.strip().split()
        if not parts or parts == [""]: continue
        if parts[0].lower() == "meta":
            if cur and cur["tokens"]: out.append(cur)
            lab = parts[2].lower() if len(parts) > 2 else None
            cur = {"uid": parts[1], "label": SENT_MAP.get(lab), "tokens": [], "tags": []}
        elif cur is not None:
            cur["tokens"].append(parts[0]); cur["tags"].append(parts[1] if len(parts) > 1 else "O")
    if cur and cur["tokens"]: out.append(cur)
    return out

def _route(name, txt, splits, labels):
    base = os.path.basename(name.lower())
    if "spanglish" in name.lower() or "spanish" in name.lower(): return
    has = lambda w: re.search(rf"(^|[^a-z]){w}([^a-z]|$)", base) is not None
    if re.search(r"^\s*meta\b", txt, re.M | re.I):
        split = "test" if has("test") else "dev" if (has("dev") or has("val") or has("valid") or has("validation")) else "trial" if has("trial") else "train"
        splits[split] += parse_conll(txt); return
    for line in txt.splitlines():
        p = re.split(r"[,\t]", line.strip())
        if len(p) >= 2 and p[-1].strip().lower() in SENT_MAP: labels[p[0].strip()] = SENT_MAP[p[-1].strip().lower()]

def load_sentimix():
    z = zipfile.ZipFile(fetch(SENTIMIX_URL, f"{DATA_DIR}/sentimix.zip")); splits, labels = collections.defaultdict(list), {}
    for n in z.namelist():
        if n.endswith("/"): continue
        raw = z.read(n)
        if n.endswith(".zip"):
            inner = zipfile.ZipFile(io.BytesIO(raw))
            for m in inner.namelist():
                if not m.endswith("/"): _route(m, inner.read(m).decode("utf-8", "ignore"), splits, labels)
        else:
            _route(n, raw.decode("utf-8", "ignore"), splits, labels)
    for k in splits:
        for ex in splits[k]:
            if ex["label"] is None and ex["uid"] in labels: ex["label"] = labels[ex["uid"]]
    data = {k: [e for e in v if e["label"] is not None] for k, v in splits.items()}
    train = data.get("train", []) + data.get("trial", []); dev, test = data.get("dev", []), data.get("test", [])
    rng = random.Random(0)
    if not test: rng.shuffle(train); test, train = train[:3000], train[3000:]
    if not dev: rng.shuffle(train); dev, train = train[:3000], train[3000:]
    return {"train": train, "dev": dev, "test": test, "labels": ["negative", "neutral", "positive"]}

# ---- Swami et al. 2018 sarcasm (GPL-3.0): blocks 'ID' + 'token  tag' lines; truth file alternates ID / YES|NO ----
def load_sarcasm():
    tw = open(fetch(SARC_BASE + "Sarcasm_tweets_with_language.txt", f"{DATA_DIR}/sarc_lang.txt"), encoding="utf-8", errors="ignore").read()
    tr = open(fetch(SARC_BASE + "Sarcasm_tweet_truth.txt", f"{DATA_DIR}/sarc_truth.txt"), encoding="utf-8", errors="ignore").read().split()
    lab = {tr[i]: (1 if tr[i + 1].upper() == "YES" else 0) for i in range(0, len(tr) - 1, 2) if tr[i + 1].upper() in ("YES", "NO")}
    exs, cur = [], None
    for line in tw.splitlines():
        s = line.strip()
        if not s or set(s) <= {"-"}: continue
        if re.fullmatch(r"\d{6,}", s):
            if cur and cur["tokens"]: exs.append(cur)
            cur = {"uid": s, "label": lab.get(s), "tokens": [], "tags": []}
        elif cur is not None:
            p = s.split()
            if len(p) >= 2: cur["tokens"].append(" ".join(p[:-1])); cur["tags"].append(p[-1])
    if cur and cur["tokens"]: exs.append(cur)
    exs = [e for e in exs if e["label"] is not None]
    rng = np.random.RandomState(0); idx = {y: rng.permutation([i for i, e in enumerate(exs) if e["label"] == y]) for y in (0, 1)}
    split = {"train": [], "dev": [], "test": []}
    for y, ii in idx.items():                         # stratified 70 / 15 / 15
        a, b = int(0.70 * len(ii)), int(0.85 * len(ii))
        split["train"] += [exs[i] for i in ii[:a]]; split["dev"] += [exs[i] for i in ii[a:b]]; split["test"] += [exs[i] for i in ii[b:]]
    split["labels"] = ["not sarcastic", "sarcastic"]
    return split

DATA = {"sentimix": load_sentimix(), "sarcasm": load_sarcasm()}
for d, D in DATA.items():
    print(d, {k: len(v) for k, v in D.items() if k != "labels"}, "| label counts (train):", collections.Counter(e["label"] for e in D["train"]),
          "| tags:", collections.Counter(t for e in D["train"] for t in e["tags"]).most_common(4))

## 2. Variant relations R and R+, and canonicalizers

In [ ]:
def load_dakshina():
    need = [f"hi.translit.sampled.{s}.tsv" for s in ("train", "dev", "test")]
    if not all(os.path.exists(f"{DATA_DIR}/{n}") for n in need):
        tpath = fetch(DAKSHINA_URL, f"{DATA_DIR}/dakshina.tar")
        with tarfile.open(tpath, "r|*") as tf:
            for m in tf:
                if os.path.basename(m.name) in need:
                    with open(f"{DATA_DIR}/{os.path.basename(m.name)}", "wb") as f: f.write(tf.extractfile(m).read())
        os.remove(tpath)
    out = {}
    for s in ("train", "dev", "test"):
        df = pd.read_csv(f"{DATA_DIR}/hi.translit.sampled.{s}.tsv", sep="\t", header=None, names=["native", "roman", "count"], quoting=3, keep_default_na=False)
        df["roman"] = df.roman.astype(str).str.lower(); df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(1); out[s] = df
    return out
LEX = load_dakshina()

def build_R(df):
    agg = df.groupby(["roman", "native"])["count"].sum().reset_index().sort_values("count", ascending=False).drop_duplicates("roman")
    r2c = dict(zip(agg.roman, agg.native)); cls = collections.defaultdict(dict)
    for r, n, c in zip(agg.roman, agg.native, agg["count"]): cls[n][r] = cls[n].get(r, 0) + float(c)
    return r2c, dict(cls)

CONS = "bcdfgjklmnpqrstvwxyz"
def rule_variants(r):
    out = set()
    if re.search(r"[aeiou]n$", r): out.add(r[:-1])                      # nahin -> nahi
    elif re.search(r"[aeiou]$", r): out.add(r + "n")                     # nahi -> nahin
    for a, b in (("aa", "a"), ("ee", "i"), ("oo", "u")):
        if a in r: out.add(r.replace(a, b))
    for b, a in (("a", "aa"), ("i", "ee"), ("u", "oo")):
        if re.search(rf"(?<![{b}]){b}(?![{b}])", r): out.add(re.sub(rf"(?<![{b}]){b}(?![{b}])", a, r, count=1))
    if re.search(rf"[{CONS}]h", r): out.add(re.sub(rf"([{CONS}])h", r"\1", r))   # bhai -> bai, kya stays
    for a, b in (("w", "v"), ("v", "w"), ("z", "j")):
        if a in r: out.add(r.replace(a, b))
    if re.search(r"[aeiou]h$", r): out.add(r[:-1])                      # yeh -> ye
    out.discard(r); return {v for v in out if len(v) >= 2}

def build_Rplus(r2c, cls):
    # attach each rule-generated spelling to the most-attested class that generates it; never merge attested classes
    claims = collections.defaultdict(list)
    for n, members in cls.items():
        tot = sum(members.values())
        for r in members:
            for g in rule_variants(r):
                if g not in r2c: claims[g].append((tot, n))
    r2c_p = dict(r2c); cls_p = {n: dict(m) for n, m in cls.items()}
    for g, cl in claims.items():
        tot, n = max(cl); r2c_p[g] = n; cls_p[n][g] = 1e-3 * tot   # tiny weight: rule variants are rare in natural text
    return r2c_p, cls_p

R_r2c, R_cls = build_R(pd.concat(LEX.values()))
Rp_r2c, Rp_cls = build_Rplus(R_r2c, R_cls)
Rtr_r2c, Rtr_cls = build_R(LEX["train"])
THREATS = {"R": (R_r2c, R_cls), "Rp": (Rp_r2c, Rp_cls)}
def canon_map(r2c, cls): return {r: max(cls[n].items(), key=lambda kv: kv[1])[0] for r, n in r2c.items()}
CMAP = {"lexR": canon_map(R_r2c, R_cls), "lexRp": canon_map(Rp_r2c, Rp_cls), "lex_train": canon_map(Rtr_r2c, Rtr_cls)}
def canon(c, t): return t if c == "raw" else CMAP[c].get(t.lower(), t)

def variants(tok, threat):
    r2c, cls = THREATS[threat]; t = tok.lower()
    if t not in r2c: return [t]
    return [r for r, _ in sorted(cls[r2c[t]].items(), key=lambda kv: -kv[1])]

# sanity: lexRp is exact for both threats, lexR exact for R
for c, th in [("lexR", "R"), ("lexRp", "R"), ("lexRp", "Rp")]:
    r2c, cls = THREATS[th]
    bad = sum(len({canon(c, v) for v in m}) > 1 for m in cls.values())
    print(f"{c} constant on {th}-classes: {bad == 0} ({bad} violations)")

def lex_stats(r2c, cls):
    s = np.array([len(m) for m in cls.values()]); return {"spellings": len(r2c), "classes": len(cls), "mean_class": float(s.mean()), "share_multi": float((s > 1).mean()), "max_class": int(s.max())}
LEXSTATS = {"R": lex_stats(R_r2c, R_cls), "R+": lex_stats(Rp_r2c, Rp_cls), "R_train": lex_stats(Rtr_r2c, Rtr_cls)}
print(json.dumps(LEXSTATS, indent=1))
for name, (r2c, cls) in [("R", (R_r2c, R_cls)), ("Rplus", (Rp_r2c, Rp_cls))]:
    pd.DataFrame([{"class_id": n, "romanization": r, "weight": w} for n, m in cls.items() for r, w in m.items()]).to_csv(f"{OUT_DIR}/lexicon/hinglish_variants_{name}.tsv", sep="\t", index=False)
print("example classes:", {w: variants(w, "Rp")[:8] for w in ["nahi", "hai", "kya", "bhai"]})

## 3. Coverage of the relations on each dataset (resource statistics)

In [ ]:
def eligible(ex, threat): return [i for i, (t, g) in enumerate(zip(ex["tokens"], ex["tags"])) if is_hin(g) and len(variants(t, threat)) > 1]
def cmi(ex):
    h = sum(is_hin(g) for g in ex["tags"]); e = sum(is_eng(g) for g in ex["tags"]); n = h + e
    return 0.0 if n == 0 else 100 * (1 - max(h, e) / n)
COV = []
for d, D in DATA.items():
    hin = [t.lower() for e in D["train"] + D["dev"] + D["test"] for t, g in zip(e["tokens"], e["tags"]) if is_hin(g)]
    for th in THREATS:
        r2c, _ = THREATS[th]
        COV.append({"dataset": d, "threat": th, "hindi_tokens": len(hin), "in_lexicon": float(np.mean([t in r2c for t in hin])),
                    "with_>=2_spellings": float(np.mean([len(variants(t, th)) > 1 for t in hin])),
                    "median_rewritable_per_test_text": float(np.median([len(eligible(e, th)) for e in D["test"]])),
                    "mean_CMI_test": float(np.mean([cmi(e) for e in D["test"]]))})
COV = pd.DataFrame(COV); COV.to_csv(f"{OUT_DIR}/tables/coverage.csv", index=False); display(COV.round(3))

## 4. Training and inference

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, balanced_accuracy_score
TOK = {}
def text_of(tokens): return " ".join(tokens)

def train_model(d, mkey, c, seed):
    D = DATA[d]; name = MODELS[mkey]
    if mkey not in TOK: TOK[mkey] = AutoTokenizer.from_pretrained(name)
    tok = TOK[mkey]; nl = len(D["labels"])
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=nl).to(DEVICE)
    X = [text_of([canon(c, t) for t in e["tokens"]]) for e in D["train"]]; Y = torch.tensor([e["label"] for e in D["train"]])
    cnt = torch.bincount(Y, minlength=nl).float(); w = (cnt.sum() / (nl * cnt.clamp_min(1))).to(DEVICE)
    lossf = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS[d] * math.ceil(len(X) / BS)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, 0.06 * steps)) * max(0.0, 1 - s / steps))
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda"); model.train()
    for ep in range(EPOCHS[d]):
        perm = np.random.RandomState(seed * 100 + ep).permutation(len(X))
        for b in range(0, len(X), BS):
            idx = perm[b:b + BS]
            enc = tok([X[i] for i in idx], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                loss = lossf(model(**enc).logits.float(), Y[idx].to(DEVICE))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return model.eval(), tok

@torch.no_grad()
def probs(model, tok, texts, bs=256):
    out = []
    for b in range(0, len(texts), bs):
        enc = tok(texts[b:b + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            out.append(torch.softmax(model(**enc).logits.float(), -1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, 1))

## 5. Evaluation of one trained model
For each test text and threat, we form the **image orbit**: for each rewritable token, the distinct canonical images of its spellings.
- If the image orbit has ≤ `CAP` members, it is enumerated **exactly**. This gives exact worst-case robust correctness and worst-case conformal coverage. For a leaky canonicalizer it is exactly the hybrid certificate, with cost equal to the orbit size.
- Larger orbits fall back to a greedy, label-aware attack, which gives an upper bound on robust accuracy; their share is reported.

In [ ]:
def image_slots(ex, c, threat):
    base = [canon(c, t) for t in ex["tokens"]]; slots = []
    for i in eligible(ex, threat):
        imgs = sorted({canon(c, v) for v in variants(ex["tokens"][i], threat)})
        if len(imgs) > 1: slots.append((i, imgs))
    return base, slots

def worst_case(ex, c, threat, model, tok):
    base, slots = image_slots(ex, c, threat); y = ex["label"]
    size = int(np.prod([len(s[1]) for s in slots])) if slots else 1
    if size <= CAP:
        texts = []
        for combo in (itertools.product(*[s[1] for s in slots]) if slots else [()]):
            tt = base.copy()
            for (i, _), im in zip(slots, combo): tt[i] = im
            texts.append(text_of(tt))
        P = probs(model, tok, texts)
        return {"exact": True, "size": size, "robust_correct": bool((P.argmax(1) == y).all()), "min_py": float(P[:, y].min())}
    tt = base.copy()                                   # greedy label-aware fallback
    for i, imgs in slots:
        trial = []
        for im in imgs:
            t2 = tt.copy(); t2[i] = im; trial.append(t2)
        P = probs(model, tok, [text_of(t) for t in trial]); tt = trial[int(np.argmin(P[:, y]))]
    P = probs(model, tok, [text_of(tt)])
    return {"exact": False, "size": size, "robust_correct": bool(P[0].argmax() == y), "min_py": float(P[0, y])}

def natural_resample(ex, c, rng):
    toks = []
    for t, g in zip(ex["tokens"], ex["tags"]):
        if is_hin(g) and len(variants(t, "R")) > 1:
            cl = R_cls[R_r2c[t.lower()]]; vs, w = zip(*cl.items()); t = rng.choices(vs, weights=w)[0]
        toks.append(canon(c, t))
    return text_of(toks)

def clopper_pearson(k, n, conf=0.95, upper=True):
    from scipy.stats import beta
    a = 1 - conf
    if upper: return 1.0 if k >= n else float(beta.ppf(1 - a, k + 1, n - k))
    return 0.0 if k <= 0 else float(beta.ppf(a, k, n - k + 1))

def evaluate(d, mkey, c, seed, model, tok):
    D = DATA[d]; test, dev = D["test"], D["dev"]; y = np.array([e["label"] for e in test]); yd = np.array([e["label"] for e in dev])
    P = probs(model, tok, [text_of([canon(c, t) for t in e["tokens"]]) for e in test]); pred = P.argmax(1)
    Pd = probs(model, tok, [text_of([canon(c, t) for t in e["tokens"]]) for e in dev])
    sc = np.sort(1 - Pd[np.arange(len(dev)), yd]); q = sc[min(math.ceil((len(dev) + 1) * (1 - ALPHA)), len(dev)) - 1]
    rng = random.Random(1000 + seed)
    Pn = probs(model, tok, [natural_resample(e, c, rng) for e in test])
    out = {"dataset": d, "model": mkey, "canon": c, "seed": seed, "n_test": len(test),
           "acc_clean": float((pred == y).mean()), "f1_clean": float(f1_score(y, pred, average="macro")), "bacc_clean": float(balanced_accuracy_score(y, pred)),
           "acc_natural": float((Pn.argmax(1) == y).mean()), "f1_natural": float(f1_score(y, Pn.argmax(1), average="macro")),
           "conf_q": float(q), "cov_clean": float((1 - P[np.arange(len(test)), y] <= q).mean()), "setsize_clean": float((1 - P <= q).sum(1).mean())}
    arrays = {"y": y, "pred": pred, "cmi": np.array([cmi(e) for e in test])}
    for th in THREATS:
        W = [worst_case(e, c, th, model, tok) for e in test]
        rc = np.array([w["robust_correct"] for w in W]); ex_ = np.array([w["exact"] for w in W]); sz = np.array([w["size"] for w in W])
        minpy = np.array([w["min_py"] for w in W])
        leak = sz > 1; k = int(leak.sum()); n = len(test)
        acc_lcb = clopper_pearson(int((pred == y).sum()), n, 0.975, upper=False); leak_ucb = clopper_pearson(k, n, 0.975, upper=True)
        out.update({f"robust_acc_{th}": float(rc.mean()), f"robust_bacc_{th}": float(np.mean([rc[y == k_].mean() for k_ in np.unique(y)])),
                    f"exact_share_{th}": float(ex_.mean()), f"orbit_median_{th}": float(np.median(sz)), f"orbit_p95_{th}": float(np.percentile(sz, 95)),
                    f"leak_rate_{th}": float(leak.mean()), f"leak_ucb_{th}": leak_ucb, f"audit_lcb_{th}": max(0.0, acc_lcb - leak_ucb),
                    f"cov_worst_{th}": float((1 - minpy <= q).mean())})
        arrays.update({f"robust_{th}": rc, f"exact_{th}": ex_, f"size_{th}": sz, f"n_elig_{th}": np.array([len(eligible(e, th)) for e in test])})
    return out, arrays

## 6. Run the grid (resumable)

In [ ]:
done = {f[:-5] for f in os.listdir(f"{OUT_DIR}/runs") if f.endswith(".json")}
for (d, m, c, s) in RUNS:
    key = f"{d}__{m}__{c}__s{s}"
    if key in done: continue
    t0 = time.time()
    model, tok = train_model(d, m, c, s)
    t1 = time.time()
    res, arr = evaluate(d, m, c, s, model, tok)
    res.update({"train_sec": t1 - t0, "eval_sec": time.time() - t1})
    json.dump(res, open(f"{OUT_DIR}/runs/{key}.json", "w"), indent=1); np.savez_compressed(f"{OUT_DIR}/runs/{key}.npz", **arr)
    print(key, {k: round(v, 3) for k, v in res.items() if k.startswith(("acc_", "f1_clean", "robust_acc", "exact_share"))}, f"{time.time()-t0:.0f}s")
    del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None
print("all runs finished:", len(os.listdir(f"{OUT_DIR}/runs")) // 2, "/", len(RUNS))

## 7. Aggregate: tables, figures, package

In [ ]:
import matplotlib.pyplot as plt
R = pd.DataFrame([json.load(open(f"{OUT_DIR}/runs/{f}")) for f in sorted(os.listdir(f"{OUT_DIR}/runs")) if f.endswith(".json")])
R.to_csv(f"{OUT_DIR}/tables/all_runs.csv", index=False)
metrics = ["acc_clean", "f1_clean", "bacc_clean", "acc_natural", "robust_acc_R", "robust_acc_Rp", "robust_bacc_R", "robust_bacc_Rp", "exact_share_R", "exact_share_Rp",
           "orbit_median_R", "orbit_p95_R", "orbit_median_Rp", "orbit_p95_Rp", "leak_rate_R", "leak_ucb_R", "audit_lcb_R", "leak_rate_Rp", "leak_ucb_Rp", "audit_lcb_Rp",
           "cov_clean", "cov_worst_R", "cov_worst_Rp", "setsize_clean"]
G = R.groupby(["dataset", "model", "canon"])[metrics]
mean, sd, n = G.mean(), G.std().fillna(0), G.size()
fmt = lambda m_, s_, k: f"{100*m_:.1f}" + (f" ± {100*s_:.1f}" if k > 1 else "")
main_cols = ["acc_clean", "f1_clean", "acc_natural", "robust_acc_R", "robust_acc_Rp", "exact_share_Rp"]
T1 = pd.DataFrame({c: [fmt(mean.loc[i, c], sd.loc[i, c], n.loc[i]) for i in mean.index] for c in main_cols}, index=mean.index)
T1.to_csv(f"{OUT_DIR}/tables/table_main.csv"); open(f"{OUT_DIR}/tables/table_main.tex", "w").write(T1.to_latex(escape=True))
hyb = mean.xs("lex_train", level="canon")[["acc_clean", "robust_acc_R", "leak_rate_R", "leak_ucb_R", "audit_lcb_R", "orbit_median_R", "orbit_p95_R",
                                           "robust_acc_Rp", "leak_rate_Rp", "audit_lcb_Rp", "orbit_p95_Rp"]]
hyb.round(4).to_csv(f"{OUT_DIR}/tables/table_hybrid_audit.csv"); open(f"{OUT_DIR}/tables/table_hybrid_audit.tex", "w").write((100 * hyb).round(1).to_latex())
conf = mean[["cov_clean", "cov_worst_R", "cov_worst_Rp", "setsize_clean"]]
conf.round(4).to_csv(f"{OUT_DIR}/tables/table_conformal.csv"); open(f"{OUT_DIR}/tables/table_conformal.tex", "w").write(conf.round(3).to_latex())
display(T1); display(hyb.round(3)); display(conf.round(3))

# Figure: robust-accuracy gap vs number of rewritable tokens (HingRoBERTa, raw vs lexRp, threat R+)
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 4))
for ax, d in zip(np.atleast_1d(axes), DATASETS):
    for c, sty in [("raw", "o-"), ("lexRp", "s--")]:
        fs = [f for f in os.listdir(f"{OUT_DIR}/runs") if f.startswith(f"{d}__hingroberta__{c}__") and f.endswith(".npz")]
        if not fs: continue
        A = [np.load(f"{OUT_DIR}/runs/{f}") for f in fs]
        ne = np.concatenate([a["n_elig_Rp"] for a in A]); gap = np.concatenate([(a["y"] == a["pred"]).astype(float) - a["robust_Rp"] for a in A])
        bins = [0, 1, 2, 3, 5, 8, 100]; lab = ["0", "1", "2", "3-4", "5-7", "8+"]; b = np.digitize(ne, bins[1:-1])
        ax.plot(lab, [100 * gap[b == k].mean() if (b == k).any() else np.nan for k in range(len(lab))], sty, label=c)
    ax.set_title(d); ax.set_xlabel("rewritable Hindi tokens (R+)"); ax.set_ylabel("clean − worst-case accuracy (pts)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures/gap_vs_tokens.pdf"); plt.savefig(f"{OUT_DIR}/figures/gap_vs_tokens.png", dpi=150); plt.show()
json.dump({"lexicon": LEXSTATS, "coverage": COV.to_dict("records"), "n_runs": len(R)}, open(f"{OUT_DIR}/tables/resource_stats.json", "w"), indent=1, default=float)
import shutil; shutil.make_archive("paper_results", "zip", OUT_DIR); print("wrote paper_results.zip  ->  upload it to the chat")